# RETFound DR — Test Only trên BRSET

Notebook này **chỉ đánh giá**, không train, không resume và không tạo pseudo-label. Quy trình: lấy `checkpoint-best.pth` từ MyDrive, tải bộ BRSET patient-wise split từ Kaggle, chỉ chạy trên thư mục `test`, rồi lưu metrics/predictions/confusion matrix về Drive.

> Bật GPU trong Colab: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Hãy chọn Runtime → Change runtime type → T4 GPU rồi chạy lại.')
print('GPU:', torch.cuda.get_device_name(0))

## 1. Gắn Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Lấy code và cài thư viện

Cell này luôn chuyển vào đúng thư mục dự án trước khi gọi module `ai`, tránh lỗi `No module named ai`.

In [ ]:
import os, subprocess, sys
from pathlib import Path

GITHUB_USERNAME = 'Bang334'
GITHUB_REPO = 'dr-diagnostic-system'
GITHUB_BRANCH = 'feat/brset-semi-patient-split'
REPO_DIR = Path('/content') / GITHUB_REPO
REPO_URL = f'https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', GITHUB_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '-b', GITHUB_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ai/grading/requirements-train.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kaggle>=2.2.2'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'Đã sẵn sàng tại {REPO_DIR} — branch {GITHUB_BRANCH}, commit {commit}')

## 3. Cấu hình duy nhất cần kiểm tra

Nếu checkpoint của bạn nằm chỗ khác, chỉ sửa `CHECKPOINT_PATH`. Kết quả sẽ được ghi vào `OUTPUT_DIR`.

In [ ]:
# ===== CẤU HÌNH =====
CHECKPOINT_PATH = Path('/content/drive/MyDrive/retfound_research/retfound_semi/semi/checkpoint-best.pth')
KAGGLE_DATASET = 'tanzinabdul/fundus-patientwise-split'
OUTPUT_DIR = Path('/content/drive/MyDrive/retfound_research/brset_test_only')
BATCH_SIZE = 2
NUM_WORKERS = 2
LIMIT_PER_CLASS = 0  # 0 = test toàn bộ; đặt 50 để chạy thử nhanh

if not CHECKPOINT_PATH.is_file():
    candidates = sorted(Path('/content/drive/MyDrive').glob('**/checkpoint-best.pth'))
    found = '\n'.join(f'  - {path}' for path in candidates) or '  (không tìm thấy file nào)'
    raise FileNotFoundError(
        f'Không tìm thấy checkpoint: {CHECKPOINT_PATH}\n'
        f'Các checkpoint-best.pth tìm thấy trên MyDrive:\n{found}\n'
        'Hãy sửa CHECKPOINT_PATH ở đầu cell này.'
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Checkpoint:', CHECKPOINT_PATH)
print('Kaggle dataset:', KAGGLE_DATASET)
print('Output:', OUTPUT_DIR)

## 4. Tải BRSET và xác minh split test

Tạo Colab Secret tên `KAGGLE_API_TOKEN`. Nếu chưa có secret, notebook sẽ yêu cầu nhập token ẩn. Dataset được cache trong `/content`, nên chạy lại cell sẽ không tải lại nếu cùng nguồn.

In [ ]:
import getpass, shutil, zipfile
from google.colab import userdata

def get_kaggle_token():
    try:
        token = userdata.get('KAGGLE_API_TOKEN')
    except Exception:
        token = getpass.getpass('Dán Kaggle API token: ').strip()
    if not token:
        raise RuntimeError('Chưa cung cấp KAGGLE_API_TOKEN')
    os.environ['KAGGLE_API_TOKEN'] = token

DATASET_DOWNLOAD_DIR = Path('/content/brset_test_dataset')
marker = DATASET_DOWNLOAD_DIR / '.kaggle_source'
if marker.is_file() and marker.read_text(encoding='utf-8').strip() == KAGGLE_DATASET:
    print('Dùng lại dataset đã tải:', DATASET_DOWNLOAD_DIR)
else:
    if DATASET_DOWNLOAD_DIR.exists():
        shutil.rmtree(DATASET_DOWNLOAD_DIR)
    DATASET_DOWNLOAD_DIR.mkdir(parents=True)
    get_kaggle_token()
    print('Đang tải Kaggle dataset:', KAGGLE_DATASET)
    subprocess.run([
        sys.executable, '-m', 'kaggle', 'datasets', 'download',
        '-d', KAGGLE_DATASET, '-p', str(DATASET_DOWNLOAD_DIR),
    ], check=True)
    archives = sorted(DATASET_DOWNLOAD_DIR.glob('*.zip'))
    if not archives:
        raise FileNotFoundError(f'Kaggle không tạo ZIP trong {DATASET_DOWNLOAD_DIR}')
    for archive_path in archives:
        print('Đang giải nén:', archive_path.name)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(DATASET_DOWNLOAD_DIR)
        archive_path.unlink()
    marker.write_text(KAGGLE_DATASET, encoding='utf-8')

# Chỉ import ai sau khi cell 2 đã os.chdir(REPO_DIR).
from ai.grading.train import find_predefined_splits, scan_classification_split
split_paths = find_predefined_splits(DATASET_DOWNLOAD_DIR)
DATASET_DIR = next(iter(split_paths.values())).parent.resolve()
test_frame = scan_classification_split(split_paths['test'], DATASET_DIR)
counts = test_frame['diagnosis'].value_counts().sort_index().to_dict()
print('Dataset root:', DATASET_DIR)
print('Test split:', split_paths['test'])
print(f'Tổng ảnh test: {len(test_frame):,}')
print('Số ảnh theo grade 0–4:', counts)

## 5. Chạy test

Lệnh dưới đây chỉ gọi `ai.grading.evaluate_test`; không có tham số train hoặc resume.

In [ ]:
def run_live(command):
    command.insert(1, '-u')
    print('Command:', ' '.join(command), flush=True)
    print('=' * 80, flush=True)
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

cmd = [
    sys.executable, '-m', 'ai.grading.evaluate_test',
    '--checkpoint', str(CHECKPOINT_PATH),
    '--dataset-dir', str(DATASET_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
]
if LIMIT_PER_CLASS > 0:
    cmd.extend(['--limit-per-class', str(LIMIT_PER_CLASS)])
run_live(cmd)

## 6. Xem kết quả

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image

metrics_path = OUTPUT_DIR / 'test_metrics.json'
predictions_path = OUTPUT_DIR / 'test_predictions.csv'
matrix_path = OUTPUT_DIR / 'confusion_matrix_normalized.png'
if not metrics_path.is_file():
    raise FileNotFoundError('Chưa có kết quả. Hãy chạy cell test ở trên trước.')
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print(json.dumps(metrics, indent=2, ensure_ascii=False))
display(pd.read_csv(predictions_path).head(10))
display(Image(filename=str(matrix_path)))
print('Kết quả đã lưu lâu dài tại:', OUTPUT_DIR)